### Import Libraries

In [2]:
import pandas as pd
import duckdb
import unicodedata
import re

### Anthropic Client

In [3]:
import anthropic
import os
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))

### DuckDB Connection

In [4]:
#import duckdb
conn = duckdb.connect('/app/data/analytics.duckdb')

### Load Data

In [5]:
df_translate = conn.execute("""
    SELECT 
        review_id,
        review_comment_message,
        review_comment_title
    FROM raw.order_reviews
    WHERE (review_comment_message IS NOT NULL AND LENGTH(review_comment_message) >= 10)
    OR (review_comment_title IS NOT NULL AND LENGTH(review_comment_title) >= 3)
""").df()

df_translate = df_translate.drop_duplicates(subset='review_id', keep='first').reset_index(drop=True)

print(f"Reviews to translate: {len(df_translate):,}")
print(f"With messages: {df_translate['review_comment_message'].notna().sum():,}")
print(f"With titles: {df_translate['review_comment_title'].notna().sum():,}")

Reviews to translate: 39,960
With messages: 38,334
With titles: 11,317


### Create Batch Request

In [6]:
# Build requests
requests = []

# Message translations - exclude lookup ones
for _, row in df_translate.iterrows():
    if pd.notna(row['review_comment_message']):
        requests.append({
            "custom_id": f"msg_{row['review_id']}",
            "params": {
                "model": "claude-haiku-4-5-20251001",
                "max_tokens": 512,
                "messages": [{
                    "role": "user",
                    "content": f"Translate this Portuguese review to English. Return only the translation, nothing else.\n\n{row['review_comment_message']}"
                }]
            }
        })

# Title translations - include all with titles
for _, row in df_translate.iterrows():
    if pd.notna(row['review_comment_title']):
        requests.append({
            "custom_id": f"title_{row['review_id']}",
            "params": {
                "model": "claude-haiku-4-5-20251001",
                "max_tokens": 256,
                "messages": [{
                    "role": "user",
                    "content": f"Translate this Portuguese review title to English. Return only the translation, nothing else.\n\n{row['review_comment_title']}"
                }]
            }
        })

print(f"Total requests: {len(requests):,}")
print(f"  Message translations: {sum(1 for r in requests if r['custom_id'].startswith('msg_')):,}")
print(f"  Title translations: {sum(1 for r in requests if r['custom_id'].startswith('title_')):,}")

Total requests: 49,651
  Message translations: 38,334
  Title translations: 11,317


### Submit Request

In [7]:
# Submit batch
batch = client.messages.batches.create(requests=requests)
#print(f"Batch ID: {batch.id}")
print(f"Status: {batch.processing_status}")

# Save batch ID
with open('/app/data/batch_id.txt', 'w') as f:
    f.write(batch.id)
print("Batch ID saved to /app/data/batch_id.txt")

Status: in_progress
Batch ID saved to /app/data/batch_id.txt
